# EDA — Candidate–Job Matching

Exploratory analysis of the candidate-job dataset. Investigates data quality, score distribution, missing values, skill vocabulary, and feature-label correlations. Findings informed all feature engineering decisions and are documented in FINDINGS.md. Input: `data/resume.csv`.

## 0. First read of the csv

df is candidate profiles matched to job positions and the match score. a lot of variables, lots of nulls, lots of detail.

* career_objective: looks like user input, so prob use for semantic similarity string?
* skills: idk? comes from a set? or user input? lets check. put them all together and figure out if they're from the same set. also, i see for example "accounts payable" and "Accounts Payables" as diff skills. dont think so. let's solve this? now? or in features.ipynb?
* educ institution: doesnt matter i think for matching. unless we use an external dataset with listed degrees/international standard like isced 
* degree name: well if from a set, we can map to potential careers through educ -> skills -> occups. but again, taxonomy talk. out of scope?
* educational results: idk? might introduce bias? but i think could be useful if normalized.. i personally think its gonna introduce bias but lets see. worried about the scope though.. this is 12 hour project.
* result types: lets print distinct. i guess we can normalize the results. (if gpa, its out of 4, if percentage, out of 100...) but again, bias?
* major: again same as degree, if we have a proper taxonomy, we can use this. for this exericse, probably not. include in slides?> is this important for the model? we start small. i prefer a simple explainable model and a description of next steps.
* company name: we dont need it i believe... 
* start dates, end dates: infer years of experience. i think we should do max end date - min start date. but in the future maybe depending on the sector/job group (horeca experience shouldnt count with CS...)
* related skills in job: list of lists.. important! flatten and if skill appears multiple times then i guess its more prominent? but also should more recent skills be more important? considerations in the future.. 
* positions: this is our main var. map to standard taxonomy? map to set of job matches?? idk
* responsibilities: idk treat as description or tasks
* job position name: is this the match?
* educational req: ideally we could extract entities here. bachelor/masters/phd = level (too advanced rn) +. entity rec: computer sc is field. can we use skillNER in this project? idk
* experience requirement: we need to categorize. use llm to categorize values or bins. check distinct values first.
* age req: same
* responsibilities.1 : is this a duplicate of responsibilities? im assuming its the responsibilities of the match. so again treat as desc or tasks for model B (semantic similairity)
* skills req: list!


-- in general i think a taxonomy is a good solution. but is gonna take a while to build. hmm. but will solve this. a skill should be matched to a predefined skill before getting matched to a job?

-- idk any outliers? nulls? duplicates? basic eda.. but i think this will require a more heuristic approach. job data is very particular, differences between proper nouns and categories are sometimes hard to detect. thats why im saying a taxonomy will solve all this. then the model can be built on top. an LLM has great potential here. this plus entity recognition for jobs, a finetuned semantic similarity model for jobs/skills/education.. for now, lets do eda, define the scope and put everything else as extra work/next steps..

# Decisions from first read

These are the decisions made from looking at the raw data before any analysis.
Nothing here is confirmed by numbers yet — that comes later in this notebook.
These are working hypotheses that may change as we explore further.

## Candidate side

**Include in candidate_doc:**
- `career_objective` — free text description of what the candidate is looking for.
  Looks like user input, quality varies. Fill nulls with empty string.
- `skills` — looks like it could be from a fixed set or free input, need to check.
  Casing is inconsistent ("Accounts Payable" vs "accounts payable" — same skill).
  Will normalise: lowercase and deduplicate before using.
- `positions` — job titles the candidate has held. Useful for matching to job title.
- `related_skils_in_job` — nested lists, needs flattening. Potentially strong signal.
- `candidate_responsibilities` — treat as description text. **Need to confirm
  this is actually candidate-side and not a duplicate of the job responsibilities.**

**Infer from dates:**
- `start_dates` + `end_dates` → infer total years of experience.
  Simple approach: max(end) - min(start). Edge cases to investigate.

**Drop:**
- `educational_institution_name` — institution name alone has no matching signal
  without an external ranking or taxonomy. Out of scope.
- `degree_names` — could be useful if mapped to a standard taxonomy (ISCED).
  Too complex for this scope. Mention as next step.
- `educational_results` / `result_types` — different grading systems
  (GPA out of 4, percentage out of 100, letter grades) are not comparable
  without normalisation. Also risk of introducing socioeconomic bias. Drop.
- `major_field_of_studies` — same as degree_names. Useful with taxonomy, not without.
- `professional_company_names` — company name alone has no matching signal.
- `extra_curricular_*` — no job-side counterpart to match against.
- `languages` / `proficiency_levels` — very sparse. No language requirement on job side.
- `address` — very sparse. No location field on job side.
- `certification_providers` / `certification_skills` — sparse. Need to check
  if there is actually usable content here before deciding.

## Job side

**Include in job_doc:**
- `job_position_name` — core job title.
- `skills_required` — key matching signal. Format looks different from candidate
  skills — needs investigation. May not be list-serialised.
- `job_responsibilities` — role description. Include as text.
- `educational_requirements` — include as raw text for now. Parsing it properly
  would need NER (SkillNER, spaCy). Mention as next step.

**Parse to structured feature:**
- `experience_requirement` — looks categorical. Need to check distinct values
  before deciding how to handle. Will try to extract integer years.

**Drop:**
- `age_requirement` — candidate age not in dataset. Also a bias risk.

## Bias note

`educational_results` and `age_requirement` are dropped partly for data quality
but also for bias reasons. Different grading systems are not comparable.
Age is a protected characteristic. Both noted in the tech note.

## Open questions to answer in this notebook

- Are `skills` from a fixed set or free text? How much variation is there?
- What is actually in `skills_required` — same format as candidate skills or different?
- Is `candidate_responsibilities` actually candidate-side or job-side?
- What are the distinct values of `experience_requirement`?
- Is `educational_results` usable at all after normalisation?
- How sparse are certifications — is there anything usable there?
- What does the score distribution look like — is it smooth or clustered?
- What does a high-scoring pair actually look like vs a low-scoring one?

These questions drive the rest of the EDA.

## 1. Raw Load

In [1]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_raw = pd.read_csv('../data/resume.csv')
df_raw.shape

(9544, 35)

In [2]:
df_raw.head(3)

,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667


In [3]:
df_raw.dtypes.to_frame()

,0
address,str
career_objective,str
skills,str
educational_institution_name,str
degree_names,str
passing_years,str
educational_results,str
result_types,str
major_field_of_studies,str
professional_company_names,str


In [4]:
# repr() to expose any invisible characters
for i, col in enumerate(df_raw.columns):
    print(f'{i:2d}  {repr(col)}')

 0  'address'
 1  'career_objective'
 2  'skills'
 3  'educational_institution_name'
 4  'degree_names'
 5  'passing_years'
 6  'educational_results'
 7  'result_types'
 8  'major_field_of_studies'
 9  'professional_company_names'
10  'company_urls'
11  'start_dates'
12  'end_dates'
13  'related_skils_in_job'
14  'positions'
15  'locations'
16  'responsibilities'
17  'extra_curricular_activity_types'
18  'extra_curricular_organization_names'
19  'extra_curricular_organization_links'
20  'role_positions'
21  'languages'
22  'proficiency_levels'
23  'certification_providers'
24  'certification_skills'
25  'online_links'
26  'issue_dates'
27  'expiry_dates'
28  '\ufeffjob_position_name'
29  'educationaL_requirements'
30  'experiencere_requirement'
31  'age_requirement'
32  'responsibilities.1'
33  'skills_required'
34  'matched_score'


## 2. Encoding & Naming Issues

Three issues visible from the column listing:
- Column 0 carries a `\ufeff` BOM prefix — the file was saved as UTF-8-with-BOM. Invisible unless you use `repr()`.
- `educationaL_requirements` — capital L.
- `experiencere_requirement` — double `re`.

Fix all three before continuing.

In [5]:
df = df_raw.copy()
df.columns = [c.lstrip('\ufeff') for c in df.columns]
df = df.rename(columns={
    'educationaL_requirements': 'educational_requirements',
    'experiencere_requirement': 'experience_requirement',
})
print(df.columns.tolist())

['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educational_requirements', 'experience_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']


In [6]:
## is responsibilities.1 just a duplicate of responsibilities?
df['responsibilities.1'].equals(df['responsibilities'])

True

In [7]:
## drop responsibilities.1 since it's a duplicate
df = df.drop(columns=['responsibilities.1'])

In [8]:
## any duplicates in the dataset?
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 0


## 2. Human Reading

What does a high-scoring pair actually look like vs a low-scoring one?

In [9]:
import ast

def _parse_list(val):
    if str(val) in ('nan', 'None', ''): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return [str(val)]

def _flatten(lst):
    out = []
    for item in lst:
        if isinstance(item, list): out.extend(str(x) for x in item if x is not None)
        elif item is not None: out.append(str(item))
    return out

def show_pair(row):
    skills      = _parse_list(row['skills'])
    positions   = _parse_list(row['positions'])
    degrees     = _parse_list(row['degree_names'])
    related     = _flatten(_parse_list(row['related_skils_in_job']))
    req_raw     = str(row['skills_required']) if str(row['skills_required']) != 'nan' else ''
    req_skills  = [s.strip() for s in req_raw.split('\n') if s.strip()]
    obj         = str(row['career_objective'])   if str(row['career_objective'])   != 'nan' else ''
    edu_req     = str(row['educational_requirements']) if str(row['educational_requirements']) != 'nan' else ''
    exp         = str(row['experience_requirement'])   if str(row['experience_requirement'])   != 'nan' else '(not specified)'
    resp        = str(row['responsibilities']) if str(row['responsibilities']) != 'nan' else ''

    print(f"  score             : {row['matched_score']:.3f}")
    print(f"  job title         : {row['job_position_name']}")
    print(f"  exp requirement   : {exp}")
    print(f"  edu requirement   : {edu_req[:100]}")
    print(f"  job skills req    : {req_skills[:6] if req_skills else '(not listed)'}")
    print(f"  cand skills       : {skills[:6]}")
    print(f"  cand positions    : {positions[:3]}")
    print(f"  cand degrees      : {degrees[:3]}")
    print(f"  related_skils_job : {related[:6]}")
    print(f"  objective         : {obj[:150] if obj else '(not provided)'}")
    print(f"  responsibilities  : {resp[:120].replace(chr(10), ' | ')}")
    print()

print('HIGH SCORE PAIRS  (matched_score > 0.85)\n')
for _, row in df[df['matched_score'] > 0.85].head().iterrows():
    print('─' * 70)
    show_pair(row)

HIGH SCORE PAIRS  (matched_score > 0.85)

──────────────────────────────────────────────────────────────────────
  score             : 0.893
  job title         : Machine Learning (ML) Engineer
  exp requirement   : At least 5 year(s)
  edu requirement   : M.Sc in Computer Science & Engineering or in any relevant discipline from a reputed University
  job skills req    : (not listed)
  cand skills       : ['Game Theory', 'Bounded Rationality', 'Cryptography', 'Algorithms', 'Social Networks', 'Analysis of Algorithms']
  cand positions    : ['Summer Intern', 'Summer Intern', 'FW Developer']
  cand degrees      : ['PhD Candidate, Computer Science', 'B.Sc., Double major in Computer Science and Management']
  related_skils_job : ['Product Development', 'Algorithm Design', 'Front-end Development', 'Theoretical Research', 'Data Experiments', 'Firmware Development']
  objective         : (not provided)
  responsibilities  : Machine Learning Leadership | Cross-Functional Collaboration | Strateg

In [10]:
## from high match pairs, how many have job skills vs not?
high_score_pairs = df[df['matched_score'] > 0.85]
high_score_pairs['has_job_skills'] = high_score_pairs['skills_required'].notna()
print("\nHIGH SCORE PAIRS - Job Skills Presence")
print(high_score_pairs['has_job_skills'].value_counts())
print("Percentage with no job skills listed: {:.2f}%".format(100 - high_score_pairs['has_job_skills'].mean() * 100))


HIGH SCORE PAIRS - Job Skills Presence
has_job_skills
True     109
False     52
Name: count, dtype: int64
Percentage with no job skills listed: 32.30%


In [11]:
## same with objective
high_score_pairs['has_objective'] = high_score_pairs['career_objective'].notna()
print("\nHIGH SCORE PAIRS - Career Objective Presence")
print(high_score_pairs['has_objective'].value_counts())
print("Percentage with no career objective listed: {:.2f}%".format(100 - high_score_pairs['has_objective'].mean() * 100))


HIGH SCORE PAIRS - Career Objective Presence
has_objective
True     107
False     54
Name: count, dtype: int64
Percentage with no career objective listed: 33.54%


observations:

* not much. except that 32% have no job skills listed and 34% no objective which is a bit annoying. 

In [12]:
print('LOW SCORE PAIRS  (matched_score < 0.3)\n')
for _, row in df[df['matched_score'] < 0.3].head(3).iterrows():
    print('─' * 70)
    show_pair(row)

LOW SCORE PAIRS  (matched_score < 0.3)

──────────────────────────────────────────────────────────────────────
  score             : 0.217
  job title         : Mechanical Engineer
  exp requirement   : 2 to 5 years
  edu requirement   : Bachelor of Science (BSc) in Mechanical Engineering, Diploma in Mechanical
  job skills req    : ['Maintenance  and Troubleshooting', 'Mechanical']
  cand skills       : ['Aderant/CMS', 'Excel', 'QuickBooks Pro', 'SQL', 'Access', 'Peachtree']
  cand positions    : ['Senior Accountant', 'Accountant', 'Financial Analyst']
  cand degrees      : ['Bachelor of Business Administration']
  related_skils_job : []
  objective         : (not provided)
  responsibilities  : Machinery Maintenance | Troubleshooting | Report Preparation | Log Maintenance

──────────────────────────────────────────────────────────────────────
  score             : 0.227
  job title         : Intern (Generative AI Engineering - 2D/3D Image Generation)
  exp requirement   : (not specif

observations:

* i think skills that are acronyms/proper nouns are challenging to map because they dont contain meaning. this is even more reason to use a taxonomy or fine tune a model to recognize this, or have a glossary.. 

## 3. Pattern Exploration

One question, one code cell, one finding.

**Q1: Are skills from a fixed controlled vocabulary or free text entered by candidates?**

In [13]:
from collections import Counter

skills_flat = []
for v in df['skills'].dropna():
    try:
        parsed = ast.literal_eval(str(v))
        skills_flat.extend(str(s) for s in parsed if s is not None)
    except Exception:
        pass

print(f'skills — total tokens: {len(skills_flat):,}')
print(f'unique (raw):          {len(set(skills_flat)):,}')
print(f'unique (lowercased):   {len(set(s.lower() for s in skills_flat)):,}')
print(f'\nTop 20 (lowercased):')
for skill, cnt in Counter(s.lower() for s in skills_flat).most_common(20):
    print(f'  {cnt:5,}  {skill}')

skills — total tokens: 206,522
unique (raw):          3,343
unique (lowercased):   2,797

Top 20 (lowercased):
  3,640  python
  3,444  machine learning
  1,736  sql
  1,568  data analysis
  1,512  deep learning
  1,494  excel
  1,204  java
  1,148  c++
  1,092  natural language processing
  1,068  sales
    980  artificial intelligence
    952  documentation
    924  data science
    924  project management
    846  accounting
    840  tableau
    840  microsoft office
    812  data mining
    812  processes
    784  c


finding: yeeeaaaa not sure. could be free text because of the accounts payable/payables differences. the most common ones are things like python, sql ... which would be quite hard to mess up in free text. so its probably free text.

**Q2: What is actually in `skills_required` — how is it formatted, and what does the vocabulary look like?**

In [14]:
req_all = []
for v in df['skills_required'].dropna():
    req_all.extend(s.strip() for s in str(v).split('\n') if s.strip())

null_rate = df['skills_required'].isna().mean()
starts_bracket = df['skills_required'].dropna().str.startswith('[').sum()
n_nonnull = df['skills_required'].notna().sum()

print(f'skills_required — null: {null_rate:.1%}')
print(f'starts with "[" (list-serialised): {starts_bracket}/{n_nonnull}')
print(f'\nRaw format (first 3 non-null values):')
for v in df['skills_required'].dropna().head(3):
    print(f'  {repr(str(v)[:150])}')
print(f'\nAfter splitting on \\n:  {len(req_all):,} tokens, {len(set(s.lower() for s in req_all)):,} unique')
print(f'\nTop 15:')
for s, c in Counter(s.lower() for s in req_all).most_common(15):
    print(f'  {c:4,}  {s}')

skills_required — null: 17.8%
starts with "[" (list-serialised): 0/7843

Raw format (first 3 non-null values):
  'Brand Promotion\nCampaign Management\nField Supervision\nMerchandising\npromotional activities\nTrade Marketing'
  'Fast typing skill\nIELTSInternet browsing & online work ability.'
  'iOS\niOS App Developer\niOS Application Development\niOS Development\nMobile apps Developer (iOS)\nNative IOS\nSwift (iOS)\nSwift UI'

After splitting on \n:  34,426 tokens, 97 unique

Top 15:
  1,025  autocad
   683  human resource management
   680  java
   342  auto cad 2d 3d
   342  civil 3d
   342  civil construction
   342  civil engineering
   342  etabs
   342  microsoft office suite
   342  ms project
   342  communication and negotiation skills
   342  internet
   342  ms office
   341  brand promotion
   341  campaign management


finding: i think this is fine. again the only problem will be matching ms project with microsoft office suite for example. i guess proper training will take care of that.

**Q3: Does `responsibilities` describe the candidate's past work or the job description?**

In [15]:
print('responsibilities — 5 rows with candidate context\n')
for _, row in df.sample(5, random_state=7).iterrows():
    try:
        pos = ast.literal_eval(str(row['positions']))
    except Exception:
        pos = []
    resp = str(row['responsibilities']) if str(row['responsibilities']) != 'nan' else ''
    print(f"  job applied for  : {row['job_position_name']}")
    print(f"  candidate held   : {[str(p) for p in pos[:2]]}")
    print(f"  responsibilities : {resp[:150]}")
    print()

responsibilities — 5 rows with candidate context

  job applied for  : Senior iOS Engineer
  candidate held   : ['Python Developer Intern']
  responsibilities : iOS Lifecycle
Requirement Analysis
Native Frameworks
iOS Development
API Integration
Technical Communication
UI Design
Performance Optimization
Featur

  job applied for  : Asst. Manager/ Manger (Administrative)
  candidate held   : ['Engineering Project Manager III', 'Field Engineer/Maintenance Support Engineer']
  responsibilities : Administrative Support
Scheduling
Filing & Documentation
Communication
Team Support
Equipment Maintenance
Information Provision
Inventory Management
T

  job applied for  : Senior iOS Engineer
  candidate held   : ['Summer Intern', 'Summer Intern']
  responsibilities : iOS Lifecycle
Requirement Analysis
Native Frameworks
iOS Development
API Integration
Technical Communication
UI Design
Performance Optimization
Featur

  job applied for  : Manager- Human Resource Management (HRM)

  candidate held 

finding: matched job responsibilities.

**Q4: What are the distinct values of `experience_requirement` — is it parseable to years?**

In [16]:
vc = df['experience_requirement'].value_counts(dropna=False)
null_rate = df['experience_requirement'].isna().mean()
print(f'experience_requirement — {df["experience_requirement"].nunique()} unique values, {null_rate:.1%} null\n')
for val, cnt in vc.items():
    print(f'  {cnt:4d}  {str(val)}')

experience_requirement — 17 unique values, 14.3% null

  1364  nan
  1024  At least 5 years
  1023  At least 1 year
  1022  At least 3 years
   682  1 to 3 years
   342  5 to 10 years
   342  1 to 2 years
   341  2 to 5 years
   341  4 to 5 years
   341  2 to 4 years
   341  At least 15 years
   341  5 to 6 years
   340  At least 5 year(s)
   340  At least 4 years
   340  3 to 5 years
   340  5 to 8 years
   340  3 to 7 years
   340  At least 2 years


finding: yeah use an llm or even just a function to bin/categorize.

**Q5: What does the score distribution look like — is it continuous or discrete?**

In [17]:
scores = df['matched_score']
print(scores.describe().round(3).to_string())
print(f'\nBelow 0.3 : {(scores < 0.3).mean():.1%}')
print(f'0.3–0.85  : {((scores >= 0.3) & (scores <= 0.85)).mean():.1%}')
print(f'Above 0.85: {(scores > 0.85).mean():.1%}')
print(f'\nUnique score values: {scores.nunique()}')
print(f'\nTop 10 most common values:')
for v, c in scores.value_counts().head(10).items():
    print(f'  {c:4d}  {v:.6f}')

count    9544.000
mean        0.661
std         0.167
min         0.000
25%         0.583
50%         0.683
75%         0.793
max         0.970

Below 0.3 : 1.9%
0.3–0.85  : 96.4%
Above 0.85: 1.7%

Unique score values: 345

Top 10 most common values:
  1470  0.850000
  1321  0.650000
   516  0.716667
   483  0.683333
   452  0.750000
   443  0.350000
   435  0.783333
   307  0.816667
   226  0.450000
   162  0.550000


finding: hmmm 345 values over 9544 rows... looks like a rule-based score stored as a float. this is gonna impact the RMSE interpretation 

In [18]:
null_rates = df.isnull().mean().sort_values(ascending=False)
null_counts = df.isnull().sum()
null_table = (
    null_rates[null_rates > 0]
    .rename('null_rate')
    .to_frame()
    .assign(null_count=null_counts)
)
null_table['null_rate'] = null_table['null_rate'].map('{:.1%}'.format)
print(null_table.to_string())

                                    null_rate  null_count
proficiency_levels                      92.7%        8844
languages                               92.7%        8844
address                                 91.8%        8760
expiry_dates                            79.0%        7536
issue_dates                             79.0%        7536
online_links                            79.0%        7536
certification_skills                    79.0%        7536
certification_providers                 79.0%        7536
extra_curricular_organization_names     64.1%        6118
role_positions                          64.1%        6118
extra_curricular_organization_links     64.1%        6118
extra_curricular_activity_types         64.1%        6118
career_objective                        50.3%        4804
age_requirement                         42.8%        4087
skills_required                         17.8%        1701
experience_requirement                  14.3%        1364
end_dates     

findings: 
* interesting. 84 candidates are missing the same data. completely empty profiles. same rows null across skills, positions, dates, degrees. less than 1% of data. drop. they carry no signal.
* 17.8% of jobs have no skills_required. high-scoring pairs are disproportionately from these jobs so missing requirements inflate scores by default (basically no skill gap since no skills are required). not a real signal. note in tech note. (another reason to use a taxonomy)
* 14.3% of jobs have no experience_requirement. same artefact as skills_required: no requirement means no penalty. scores inflate by default. when parsing experience gap as a feature, treat null as 'no constraint' not as missing data ??
* certification-related columns (providers, skills, issue_dates, expiry_dates, online_links) all share exactly 79% nulls bc they're properties of the same object. 79% of candidates have no certifications. Already decided to drop these and now confirmed by null analysis.

In [19]:
import re

def parse_experience(s):
    if not isinstance(s, str):
        return None
    s = s.lower().strip()
    # "X to Y years" → midpoint
    m = re.search(r"(\d+(?:\.\d+)?)\s*to\s*(\d+(?:\.\d+)?)", s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    # "at least X" or "X year(s)"
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if m:
        return float(m.group(1))
    return None

parsed = df['experience_requirement'].apply(parse_experience)
n_nonnull = df['experience_requirement'].notna().sum()
n_parsed  = parsed.notna().sum()

print(f"non-null rows   : {n_nonnull}")
print(f"successfully parsed: {n_parsed}  ({n_parsed/n_nonnull:.1%} of non-null)")
print()
print(parsed.value_counts(dropna=False).sort_index().to_string())

non-null rows   : 8180
successfully parsed: 8180  (100.0% of non-null)

experience_requirement
1.0     1023
1.5      342
2.0     1022
3.0     1363
3.5      341
4.0      680
4.5      341
5.0     1704
5.5      341
6.5      340
7.5      342
15.0     341
NaN     1364


In [20]:
import ast
from collections import Counter

def _parse(val):
    if str(val) in ('nan','None',''): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return []

rt_flat = [str(v) for row in df['result_types'].apply(_parse) for v in row if v is not None]
print('result_types value_counts:')
for val, cnt in Counter(rt_flat).most_common():
    print(f'  {cnt:5d}  {val}')

# 10 educational_results paired with their result_type
print()
print('educational_results — one example per result_type:')
seen = set()
shown = 0
for _, row in df.iterrows():
    rts = _parse(row['result_types'])
    ers = _parse(row['educational_results'])
    for rt, er in zip(rts, ers):
        rt, er = str(rt), str(er)
        if rt not in seen and er not in ('N/A','nan','None',''):
            print(f'  result_type={rt:25s}  result={er}')
            seen.add(rt)
            shown += 1
    if shown >= 10:
        break

result_types value_counts:
   8388  N/A
   1774  GPA
    140  Percentage
     84  CGPA
     56  Honors
     28  WAM
     28  Major GPA
     28  Maintained an A average every quarter
     28  Obtained highest honors certificate every quarter

educational_results — one example per result_type:
  result_type=None                       result=3.84
  result_type=GPA                        result=2.50
  result_type=CGPA                       result=9.35 (Till 4 semesters)
  result_type=Percentage                 result=94%
  result_type=N/A                        result=Gold Medalist
  result_type=WAM                        result=79.42
  result_type=Honors                     result=Cum Laude


  result_type=Major GPA                  result=3.45
  result_type=Maintained an A average every quarter  result=A average
  result_type=Obtained highest honors certificate every quarter  result=3.7


findings: too many incomparable systems — gpa/4, cgpa/10, percentage/100, honors, wam... normalising is possible (4/4 = 100%) but out of scope for now. dropping. bias risk too. mention normalisation as next step.

In [21]:
all_related = []
raw_examples = []
for val in df['related_skils_in_job'].dropna():
    try:
        parsed = ast.literal_eval(str(val))
    except Exception:
        continue
    if not isinstance(parsed, list):
        continue
    if len(raw_examples) < 3:
        raw_examples.append(parsed)
    for item in parsed:
        if isinstance(item, list):
            all_related.extend(str(x) for x in item if x is not None)
        elif item is not None:
            all_related.append(str(item))

print(f'related_skils_in_job — {len(all_related):,} values when flattened')
print('\ntop 30:')
for skill, cnt in Counter(s.lower() for s in all_related).most_common(30):
    print(f'  {cnt:5d}  {skill}')

print('\n3 raw examples (nested structure):')
for ex in raw_examples:
    print(f'  {ex}')

related_skils_in_job — 86,264 values when flattened

top 30:
   1484  machine learning
   1036  project management
    908  sales
    868  troubleshooting
    756  data analysis
    728  python
    710  customer service
    600  marketing
    578  accounting
    560  maintenance
    552  leadership
    504  budgeting
    504  testing
    476  financial statements
    476  communication
    448  c++
    420  quality assurance
    392  automation
    364  documentation
    336  installation
    336  nlp
    308  accounts receivable
    308  training
    308  sql
    308  general ledger
    284  research
    280  account reconciliation
    280  time management
    280  audit
    280  accounts payable

3 raw examples (nested structure):
  [['Big Data']]
  [['Data Analysis', 'Business Analysis', 'Machine Learning']]
  [['Unified Payment Interface', 'Risk Prediction', 'Big Data', 'Spark', 'PySpark']]


cleaner vocabulary than candidate skills but same synonym problem (nlp vs natural language processing will never match on surface form). flattening works. this is exactly why we need embeddings or a taxonomy. include in candidate_doc, accept the limitation.

In [22]:
years_flat = []
for val in df['passing_years'].dropna():
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            years_flat.extend(str(y) for y in parsed if y is not None)
    except Exception:
        pass

print(f'passing_years — {len(years_flat):,} values across all candidates')
print()
for val, cnt in sorted(Counter(years_flat).items()):
    print(f'  {cnt:5d}  {val}')

passing_years — 14,622 values across all candidates

     28  01/2011
     28  02/2002
     28  02/2010
     28  02/2015
     28  02/2019
     28  03/2002
     28  05/2002
     28  05/2013
     28  05/2015
     28  06/2013
     28  08/2002
     28  08/2003
     28  09/2000
     28  09/2009
     28  1
     28  12/2014
     28  1977
     28  1978
     56  1980
     28  1981
     84  1983
     28  1984
     28  1985
     28  1986
     56  1987
     56  1988
     28  1989
    128  1990
    104  1991
     84  1992
    140  1993
     56  1994
     28  1994-05-15
     62  1995
    112  1996
    112  1997
     56  1999
     84  2000
    174  2001
     28  2001-12-15
    140  2002
    196  2003
    336  2004
     28  2004-12-18
    196  2005
    336  2006
    196  2007
    258  2008
    168  2009
    496  2010
    236  2011
    224  2012
    236  2013
    336  2014
    308  2015
    308  2016
    728  2017
    784  2018
   2128  2019
   1512  2020
    560  2021
     84  2023
     28  2025
     

messy formats, goes back to 1977. not using this directly. we infer experience from start_dates/end_dates instead. drop passing_years.

In [23]:
resp = df['responsibilities'].dropna().astype(str)
lengths = resp.str.len()
print(f'responsibilities — {len(resp):,} non-null rows')
print(f'char length:  min={lengths.min()}  mean={lengths.mean():.0f}  max={lengths.max()}')
print()
print('5 raw values:')
for v in resp.head(5):
    print(repr(v[:250]))
    print()

responsibilities — 9,544 non-null rows
char length:  min=72  mean=217  max=587

5 raw values:
'Technical Support\nTroubleshooting\nCollaboration\nDocumentation\nSystem Monitoring\nSoftware Deployment\nTraining & Mentorship\nIndustry Trends\nField Visits\n\n\n\n\n'

'Machine Learning Leadership\nCross-Functional Collaboration\nStrategy Development\nML/NLP Infrastructure\nPrototype Transformation\nML System Design\nAlgorithm Research\nApplication Development\nDataset Selection\nML Testing\nStatistical Analysis\nR&D in ML/NLP'

'Trade Marketing Executive\nBrand Visibility, Sales Targets\nField Marketing, Campaigns, Product Distribution\nBrand Head\nExcel, KPIs Tracking'

'Apparel Sourcing\nQuality Garment Sourcing\nReliable Partner\nBuyer/Vendor Communication'

'iOS Lifecycle\nRequirement Analysis\nNative Frameworks\niOS Development\nAPI Integration\nTechnical Communication\nUI Design\nPerformance Optimization\nFeature Collaboration\nBug Fixing\nCode Translation\nHigh-Performance Developm

job-side confirmed. zero nulls, newline-separated, mean 217 chars. clean enough to include in job_doc after replacing newlines with spaces.

In [24]:
from collections import defaultdict

skills_flat = []
for val in df['skills'].dropna():
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            skills_flat.extend(str(s) for s in parsed if s is not None)
    except Exception:
        pass

case_groups = defaultdict(Counter)
for s in skills_flat:
    case_groups[s.lower().strip()][s] += 1

multi = {k: v for k, v in case_groups.items() if len(v) > 1}
ranked = sorted(multi.items(), key=lambda x: -sum(x[1].values()))

print(f'Skill tokens with casing variants: {len(multi):,} of {len(case_groups):,} unique tokens')
print()
print(f'{"token":<35}  {"total":>5}  variants')
print('─' * 80)
for token, variants in ranked[:15]:
    total = sum(variants.values())
    parts = '  '.join(f'{repr(v)}×{c}' for v, c in variants.most_common())
    print(f'{token:<35}  {total:5d}  {parts}')

Skill tokens with casing variants: 486 of 2,797 unique tokens

token                                total  variants
────────────────────────────────────────────────────────────────────────────────
python                                3640  'Python'×3612  'python'×28
machine learning                      3444  'Machine Learning'×3220  'Machine learning'×196  'machine learning'×28
sql                                   1736  'SQL'×1708  'Sql'×28
data analysis                         1568  'Data Analysis'×1456  'data analysis'×84  'Data analysis'×28
deep learning                         1512  'Deep Learning'×1316  'Deep learning'×196
excel                                 1494  'Excel'×1438  'excel'×56
java                                  1204  'Java'×1148  'JAVA'×56
natural language processing           1092  'Natural Language Processing'×952  'Natural language Processing'×112  'Natural language processing'×28
sales                                 1068  'Sales'×536  'sales'×532
artificia

486 of 2797 unique tokens have casing variants. python/Python, SQL/Sql, JAVA/Java. lowercasing fixes this. do in feature engineering, not here.

## Visualisations

Three plots that summarise the core EDA findings for the presentation.

In [25]:
# Visualisation 1 — score distribution with quasi-discrete annotation
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

scores = df['matched_score']
top_vals = scores.value_counts().head(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(scores, bins=60, color='#4a9aba', edgecolor='white', linewidth=0.3)
axes[0].axvline(scores.mean(), color='#e94560', linestyle='--', lw=1.5, label=f'mean={scores.mean():.3f}')
axes[0].axvline(scores.median(), color='#f5a623', linestyle='--', lw=1.5, label=f'median={scores.median():.3f}')
for v in [0.65, 0.85]:
    axes[0].axvline(v, color='#aaa', linestyle=':', lw=0.8)
    axes[0].text(v+0.005, axes[0].get_ylim()[1]*0.85, f'{v}', fontsize=8, color='#666')
axes[0].set_xlabel('matched_score'); axes[0].set_ylabel('Count')
axes[0].set_title('Score distribution')
axes[0].legend(fontsize=9)
axes[0].annotate(f'{scores.nunique()} unique values\n0.85 and 0.65 = {((scores==0.85)|(scores==0.65)).mean():.0%} of rows',
                 xy=(0.02, 0.75), xycoords='axes fraction', fontsize=9, color='#444')

sorted_s = np.sort(scores.values)
cdf = np.arange(1, len(sorted_s)+1) / len(sorted_s)
axes[1].plot(sorted_s, cdf, color='#4a9aba', lw=1.5)
for t in [0.3, 0.65, 0.85]:
    pct = (scores < t).mean()
    axes[1].axvline(t, color='grey', linestyle=':', lw=0.8)
    axes[1].text(t+0.01, 0.08, f'{t}\n({pct:.0%})', fontsize=8, color='#666')
axes[1].set_xlabel('matched_score'); axes[1].set_ylabel('Cumulative %')
axes[1].set_title('CDF — thin tails (1.9% below 0.3, 1.7% above 0.85)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('../outputs/eda_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Top 5 most common scores: {dict(scores.value_counts().head(5).round(6))}')

Top 5 most common scores: {0.85: np.int64(1470), 0.65: np.int64(1321), 0.716666667: np.int64(516), 0.683333333: np.int64(483), 0.75: np.int64(452)}


/tmp/claude-501/ipykernel_29541/1205790681.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
# Visualisation 2 — vocabulary mismatch: candidate vs job skill vocabulary
import ast
from collections import Counter

def safe_parse(val):
    if not val or str(val).strip() in ('','[]','nan'): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except: return []

cand_vocab = {str(s).lower().strip()
              for lst in df['skills'].apply(safe_parse) for s in lst if s}
job_vocab  = set()
for lst in df['skills_required'].dropna():
    for s in str(lst).split('\n'):
        s = s.strip()
        if s: job_vocab.add(s.lower())

overlap = cand_vocab & job_vocab

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: vocabulary sizes
cats  = ['Candidate\nskills', 'Job\nskills_required', 'Overlap']
vals  = [len(cand_vocab), len(job_vocab), len(overlap)]
cols  = ['#4a9aba', '#e06c5a', '#f5a623']
bars  = axes[0].bar(cats, vals, color=cols, width=0.5)
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+30, str(v),
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Unique tokens')
axes[0].set_title('Vocabulary sizes — structural mismatch')
axes[0].annotate('Only 31 tokens shared.\nJob side uses composite phrases\nas single tokens.',
                 xy=(0.6, 0.6), xycoords='axes fraction', fontsize=10, color='#333',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff8e1', alpha=0.8))

# Jaccard distribution (most are zero)
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
feat2 = pd.read_csv('../outputs/features.csv')
jac = feat2['skill_jaccard']
axes[1].hist(jac[jac > 0], bins=30, color='#4a9aba', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('skill_jaccard (nonzero only)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Jaccard > 0 for only {(jac>0).mean():.1%} of pairs')
axes[1].annotate(f'{(jac==0).mean():.1%} of pairs\nhave Jaccard = 0\n(no token overlap)',
                 xy=(0.55, 0.6), xycoords='axes fraction', fontsize=11, color='#e94560')

plt.tight_layout()
plt.savefig('../outputs/eda_vocab_mismatch.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Candidate vocab: {len(cand_vocab):,}  Job vocab: {len(job_vocab):,}  Overlap: {len(overlap)}')
print(f'Jaccard = 0: {(jac==0).mean():.1%}  Fuzzy (threshold 90): {(feat2.fuzzy_skill_jaccard>0).mean():.1%}')

Candidate vocab: 2,797  Job vocab: 97  Overlap: 31
Jaccard = 0: 94.2%  Fuzzy (threshold 90): 6.1%


/tmp/claude-501/ipykernel_29541/1407986550.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
# Visualisation 3 — feature-label Spearman correlations
from scipy.stats import spearmanr

feat3 = pd.read_csv('../outputs/features.csv')
features_to_check = {
    'title_semantic_sim': feat3['title_semantic_sim'],
    'exp_deficit (neg)':  -feat3['exp_deficit'],
    'is_fresher (neg)':   -feat3['is_fresher'],
    'years_experience':   feat3['years_experience'],
    'exp_surplus':        feat3['exp_surplus'],
    'fuzzy_skill_cov':    feat3['fuzzy_skill_coverage'],
    'skill_coverage':     feat3['skill_coverage'],
    'skills_req_count':   feat3['skills_required_count'],
    'edu_match':          feat3['edu_match'],
}
y = feat3['matched_score']
results = {name: spearmanr(vals, y)[0] for name, vals in features_to_check.items()}
results_sorted = sorted(results.items(), key=lambda x: x[1])

names = [r[0] for r in results_sorted]
vals  = [r[1] for r in results_sorted]
cols  = ['#e06c5a' if v < 0 else '#4a9aba' for v in vals]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(names, vals, color=cols)
ax.axvline(0, color='black', lw=0.8)
for b, v in zip(bars, vals):
    ax.text(v + (0.003 if v >= 0 else -0.003), b.get_y()+b.get_height()/2,
            f'{v:+.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlabel('Spearman r with matched_score')
ax.set_title('Individual feature signal strength')
ax.annotate('Label R² from all structured\nfeatures combined: only 9.7%\n→ label is mostly text-driven',
            xy=(0.65, 0.15), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff3cd', alpha=0.9))
plt.tight_layout()
plt.savefig('../outputs/eda_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/claude-501/ipykernel_29541/358692130.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
